In [1]:
import pandas as pd
import polars as pl
import util
from pathlib import Path

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
transit_jobs_access = pd.read_csv(util.output_path / 'access/transit_jobs_access.csv', 
                                  usecols=['geography', 'value', 'geography_group'])
walk_bike_jobs_access = pd.read_csv(util.output_path / 'access/walk_bike_jobs_access.csv', 
                                  usecols=['geography_value', 'jobs_1_mile_walk', 'jobs_3_mile_bike', 'geography_group']).\
                                  rename(columns={'geography_value': 'geography'})

parcel_emp = util.get_parcels_urbansim_data(inc_geog=True)

# jobs access in equity geographies
equity_geogs = util.summary_config['equity_geogs']

## Jobs Accessible within 45 Minutes of Transit

In [3]:
def process_access_data(jobs_access):
    df_access = jobs_access.copy()
    # rename region
    df_access.loc[jobs_access['geography_group'] == 'region', 'geography'] = 'Region'
    # rename rgc
    df_access.loc[jobs_access['geography_group'] == 'rgc_binary', 'geography'] = ['Not in RGC', 'In RGC']
    # rename regional geography
    df_access.loc[jobs_access['geography_group'] == 'rg_proposed', 'geography_group'] = 'RegionalGeogName'

    equity_geogs = list(util.equity_geog_dict.keys())
    df_access_equity = df_access.loc[
        df_access['geography_group'].isin(equity_geogs)].copy()
    
    df_access_equity['geography'] = df_access_equity['geography'].map({"0.0": 'Below Regional Average', 
                                                                       "1.0": 'Above Regional Average', 
                                                                       "2.0": 'Higher Share of Equity Population'}
                                                                       )
    df_access_equity['geography_group'] = df_access_equity['geography_group'].map(util.equity_geog_dict)

    # df_access_equity_geogs['geography'] = df_access_equity_geogs['geography_group']
    # df['geography'] = "NOT in " + df['geography_group']

    # df_access_equity_geogs = pd.concat([df_access_equity_geogs, df], ignore_index=True)
    # df_access_equity_geogs['geography_group'] = 'Equity Geography'

    return df_access, df_access_equity


df_access_t, df_access_equity_t = process_access_data(transit_jobs_access)
df_access_bp, df_access_equity_bp = process_access_data(walk_bike_jobs_access)
tot_jobs = parcel_emp['emptot_p'].sum()

In [4]:
def job_access_geog(access_table, geog, format=True):
    df = access_table.loc[access_table['geography_group'].isin([geog, 'region'])].\
        rename(columns={'value': 'Jobs within 45-minute Transit Commute'}).\
        drop(columns=['geography_group']).\
        set_index('geography')
    
    df_jobs = parcel_emp.groupby(geog)['emptot_p'].sum()
    df_jobs['Region'] = tot_jobs
    df = pd.concat([df, df_jobs.rename('Total Jobs within Geography')], axis=1)
    df = df.loc[df.index != 'Outside Region'].copy()

    df['% All Jobs'] = df['Jobs within 45-minute Transit Commute'] / tot_jobs
    df['% of Jobs in Geography'] = df['Jobs within 45-minute Transit Commute'] / df['Total Jobs within Geography']

    if format:
        return df.style.format('{:,.1%}', subset=['% All Jobs','% of Jobs in Geography']).\
            format('{:,.0f}', subset=['Jobs within 45-minute Transit Commute','Total Jobs within Geography'])
    else:
        return df


In [5]:
df = job_access_geog(df_access_t,'CountyName')
# df = df[df.index != 'Outside Region']
df

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
King,"285,926","1,717,963",10.9%,16.6%
Kitsap,"14,479","124,951",0.6%,11.6%
Pierce,"39,995","392,878",1.5%,10.2%
Snohomish,"62,555","386,170",2.4%,16.2%
Region,"175,742","2,621,962",6.7%,6.7%


In [6]:
df_rgc = job_access_geog(df_access_t,'rgc_binary')
df_rgc

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"175,742","2,621,962",6.7%,6.7%
Not in RGC,"127,438","1,628,934",4.9%,7.8%
In RGC,"463,248","993,028",17.7%,46.7%


In [7]:
df = job_access_geog(df_access_t,'GrowthCenterName')

df

,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"175,742","2,621,962",6.7%,6.7%
Auburn,"121,542","7,752",4.6%,"1,567.9%"
Bellevue,"489,825","81,949",18.7%,597.7%
Bothell Canyon Park,"42,446","16,186",1.6%,262.2%
Bremerton,"54,982","25,333",2.1%,217.0%
Burien,"174,740","6,168",6.7%,"2,833.0%"
Everett,"135,821","30,497",5.2%,445.4%
Federal Way,"152,307","5,482",5.8%,"2,778.3%"
Greater Downtown Kirkland,"422,522","16,808",16.1%,"2,513.8%"
Issaquah,"19,993","12,224",0.8%,163.6%


In [8]:
job_access_geog(df_access_t,'RegionalGeogName')


,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
Region,"175,742","2,621,962",6.7%,6.7%
Cities and Towns,"15,428","121,717",0.6%,12.7%
Core Cities,"95,620",nan,3.6%,nan%
High Capacity Transit Communities,"73,800",nan,2.8%,nan%
Metropolitan Cities,"393,496",nan,15.0%,nan%
Rural Areas,"3,210",nan,0.1%,nan%
Urban Unincorporated Areas,"17,660",nan,0.7%,nan%


In [9]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = job_access_geog(df_access_equity_t, label, format=False)
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'index': 'EFA Type'}, inplace=True)
# df
df.loc[df['EFA Type']!="Region"][['Group', 'EFA Type', 'Jobs within 45-minute Transit Commute',
    'Total Jobs within Geography', '% All Jobs', '% of Jobs in Geography']].style.\
        format('{:,.1%}', subset=['% All Jobs','% of Jobs in Geography']).\
        format('{:,.0f}', subset=['Jobs within 45-minute Transit Commute','Total Jobs within Geography'])

,Group,EFA Type,Jobs within 45-minute Transit Commute,Total Jobs within Geography,% All Jobs,% of Jobs in Geography
0,People of Color,Below Regional Average,"147,373","995,215",5.6%,14.8%
1,People of Color,Above Regional Average,"205,562","941,251",7.8%,21.8%
2,People of Color,Higher Share of Equity Population,"210,255","685,495",8.0%,30.7%
4,Income,Below Regional Average,"172,412","1,276,851",6.6%,13.5%
5,Income,Above Regional Average,"172,159","747,577",6.6%,23.0%
6,Income,Higher Share of Equity Population,"193,479","597,533",7.4%,32.4%
8,LEP,Below Regional Average,"174,795","1,397,419",6.7%,12.5%
9,LEP,Above Regional Average,"186,636","628,397",7.1%,29.7%
10,LEP,Higher Share of Equity Population,"165,342","596,145",6.3%,27.7%
12,Disability,Below Regional Average,"194,162","1,273,412",7.4%,15.2%


## Average Jobs Accessible within 1 Mile Walk and 3 Mile Bike
Note that this is not using the bike network, but is instead using the all-streets network.

Average accessible jobs are weighted averages based on parcel household population.

In [10]:
def bp_job_access_geog(access_table,geog):
    df = access_table.loc[access_table['geography_group'].isin(['region', geog])].\
        rename(columns={'jobs_1_mile_walk': 'Jobs within 1-mile Walk',
                        'jobs_3_mile_bike': 'Jobs within 3-mile Bike'}).\
        drop(columns=['geography_group']).\
        set_index('geography')

    df['% Total Jobs (1-mile walk)'] = df['Jobs within 1-mile Walk'].apply(lambda x: f'{x / tot_jobs * 100:,.1f}' + '%')
    df['% Total Jobs (3-mile bike)'] = df['Jobs within 3-mile Bike'].apply(lambda x: f'{x / tot_jobs * 100:,.1f}' + '%')

    return df

In [11]:
df = bp_job_access_geog(df_access_bp,'CountyName')
df = df[df.index != 'Outside Region']
df

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
King,"28,051","109,151",1.1%,4.2%
Kitsap,"1,894","11,172",0.1%,0.4%
Pierce,"5,070","27,186",0.2%,1.0%
Snohomish,"3,671","26,119",0.1%,1.0%
Region,"17,032","70,310",0.6%,2.7%


In [12]:
df_rgc = bp_job_access_geog(df_access_bp,'rgc_binary')
df = bp_job_access_geog(df_access_bp,'GrowthCenterName')

pd.concat([df_rgc, df.loc[~df.index.isin(['Region','Not in RGC'])]], axis=0)

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
Region,"17,032","70,310",0.6%,2.7%
Not in RGC,"3,554","41,269",0.1%,1.6%
In RGC,"97,251","243,164",3.7%,9.3%
Auburn,"16,875","50,350",0.6%,1.9%
Bellevue,"88,623","144,717",3.4%,5.5%
Bothell Canyon Park,"10,698","28,558",0.4%,1.1%
Bremerton,"15,603","42,947",0.6%,1.6%
Burien,"7,753","18,736",0.3%,0.7%
Everett,"29,888","65,356",1.1%,2.5%


In [13]:
bp_job_access_geog(df_access_bp,'RegionalGeogName')

,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
geography,,,,
Region,"17,032","70,310",0.6%,2.7%
Cities and Towns,"1,126","10,735",0.0%,0.4%
Core Cities,"5,494","37,810",0.2%,1.4%
High Capacity Transit Communities,"1,996","20,054",0.1%,0.8%
Metropolitan Cities,"43,999","161,836",1.7%,6.2%
Rural Areas,171,"2,860",0.0%,0.1%
Urban Unincorporated Areas,569,"8,792",0.0%,0.3%


In [14]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = bp_job_access_geog(df_access_equity_bp, label)
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df.rename(columns={'geography': 'EFA Type'}, inplace=True)
df[['Group', 'EFA Type', 'Jobs within 1-mile Walk',	'Jobs within 3-mile Bike',	'% Total Jobs (1-mile walk)','% Total Jobs (3-mile bike)']]

,Group,EFA Type,Jobs within 1-mile Walk,Jobs within 3-mile Bike,% Total Jobs (1-mile walk),% Total Jobs (3-mile bike)
0,People of Color,Below Regional Average,"7,275","49,936",0.3%,1.9%
1,People of Color,Above Regional Average,"28,372","95,805",1.1%,3.7%
2,People of Color,Higher Share of Equity Population,"27,026","88,056",1.0%,3.4%
3,Income,Below Regional Average,"14,573","67,011",0.6%,2.6%
4,Income,Above Regional Average,"20,013","72,715",0.8%,2.8%
5,Income,Higher Share of Equity Population,"20,987","78,213",0.8%,3.0%
6,LEP,Below Regional Average,"16,530","72,505",0.6%,2.8%
7,LEP,Above Regional Average,"22,289","72,486",0.9%,2.8%
8,LEP,Higher Share of Equity Population,"12,168","59,441",0.5%,2.3%
9,Disability,Below Regional Average,"14,971","72,185",0.6%,2.8%


## Intersection Density

In [15]:
buffered_parcels = pl.read_csv(util.output_path / 'landuse/buffered_parcels.txt', 
                               separator=' ',
                               columns=['parcelid','nodes3_2','nodes4_2','hh_p']).to_pandas()


list_cols = ['ParcelID','CountyName','GrowthCenterName','rg_proposed'] + equity_geogs
parcel_geog = util.get_parcel_geog()[list_cols]

In [16]:
df_intersection = buffered_parcels.merge(parcel_geog, left_on='parcelid', right_on='ParcelID')

# Total intersections within 1/2 mile buffer
df_intersection['intersections_wt'] = (df_intersection['nodes3_2'] + df_intersection['nodes4_2']) * df_intersection['hh_p']

In [17]:
def intersection_density(geog, map=False):
    df = df_intersection.groupby(geog)[['intersections_wt', 'hh_p']].sum().reset_index()
    df['Intersections'] = df['intersections_wt']/df['hh_p']

    if map:
        df[geog] = df[geog].astype('int').map({0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}
                                )
    
    return df[[geog] + ['Intersections']]

In [18]:
df = intersection_density('CountyName')
df = df[df['CountyName']!='Outside Region']
df

,CountyName,Intersections
0,King,163
1,Kitsap,53
3,Pierce,86
4,Snohomish,78


In [19]:
intersection_density('GrowthCenterName')

,GrowthCenterName,Intersections
0,Auburn,189
1,Bellevue,274
2,Bothell Canyon Park,71
3,Bremerton,166
4,Burien,179
5,Everett,162
6,Federal Way,133
7,Greater Downtown Kirkland,165
8,Issaquah,96
9,Kent,214


In [20]:
intersection_density('rg_proposed')

,rg_proposed,Intersections
0,Cities and Towns,64
1,Core Cities,108
2,High Capacity Transit Communities,86
3,Metropolitan Cities,210
4,Rural Areas,20
5,Urban Unincorporated Areas,54


In [21]:
df = pd.DataFrame()
for col, label in util.equity_geog_dict.items():
    _df = intersection_density(col).rename(columns={col: "Group"})
    _df['Group'] = label
    df = pd.concat([df, _df])
df = df.reset_index()
df['EFA Type'] = df['index'].map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })
df[['Group', 'EFA Type', 'Intersections']]

,Group,EFA Type,Intersections
0,People of Color,Below Regional Average,104
1,People of Color,Above Regional Average,147
2,People of Color,Higher Share of Equity Population,145
3,Income,Below Regional Average,116
4,Income,Above Regional Average,127
5,Income,Higher Share of Equity Population,147
6,LEP,Below Regional Average,121
7,LEP,Above Regional Average,131
8,LEP,Higher Share of Equity Population,126
9,Disability,Below Regional Average,125
